# Week 1: behavior run on Gemma-2-2B
Runtime: **T4 GPU**, High-RAM recommended. Secrets (key icon): `HF_TOKEN`, and `GITHUB_TOKEN` if you clone the private repo. Run the cells top to bottom.

## 1. Get the code
Either clone with a GitHub token (a fine-grained token with read-only **Contents** access to this repo), or upload a zip of the repo in the file panel and unzip it. The cell skips cloning if the folder already exists.

In [ ]:
import os, subprocess
if not os.path.exists("textbook-vs-data"):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/acrodrig98/textbook-vs-data.git"], check=True)
    # keep the token out of the clone's saved settings
    subprocess.run(["git", "-C", "textbook-vs-data", "remote", "set-url", "origin",
                    "https://github.com/acrodrig98/textbook-vs-data.git"], check=True)
%cd textbook-vs-data
!git log --oneline -3

## 2. Install and log in
Only what the behavior run needs; circuit-tracer comes in Week 2.

In [ ]:
!pip install -q transformers==4.57.3 huggingface_hub==0.36.2 pytest
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

## 3. Tests and prompts

In [ ]:
!python -m pytest -q tests/test_generate.py tests/test_eval.py
!python -m prompts.generate --out data/week1.jsonl --preview 1

## 4. Quick check, then the full run
The quick check scores 64 prompts. If it looks fine, the full run takes several minutes.

In [ ]:
!HF_HUB_DISABLE_PROGRESS_BARS=1 python -m eval.behavior --model google/gemma-2-2b --limit 64 --out results/quickcheck.jsonl

In [ ]:
!HF_HUB_DISABLE_PROGRESS_BARS=1 python -m eval.behavior --model google/gemma-2-2b --prompts data/week1.jsonl

## 5. Phase diagrams and go/no-go

In [ ]:
!python -m eval.analyze results/week1_gemma-2-2b.jsonl
from IPython.display import Image, display
display(Image("figures/week1_gemma-2-2b_phase.png"))
display(Image("figures/week1_gemma-2-2b_levels.png"))

## 6. Download the results
Commit these to the repo from your Mac.

In [ ]:
!zip -q -r week1_results.zip results/week1_gemma-2-2b* figures/week1_gemma-2-2b*
from google.colab import files
files.download("week1_results.zip")